### Part 1
Fetch RSS Feed into the context

In [24]:
import requests
import pandas as pd

In [25]:
ADVERTS_RSS_URL = "https://www.adverts.ie/for-sale/rss/$"

In [26]:
def fetch_rss_feed(url: str) -> str:
  """Fetch raw XML/RSS content with a descriptive User-Agent and timeout."""
  headers = {
      "User-Agent": (
          "AcademicDataPipeline/1.0 (Student AI Project; contact via 20101674@mydbs.ie)"
      ),
      "Accept": "application/rss+xml, application/xml, text/xml;q=0.9, */*;q=0.8",
  }

  response = requests.get(url, headers=headers, timeout=15)
  response.raise_for_status()  # Raises HTTPError if occured
  return response.text

In [27]:
raw_xml = fetch_rss_feed(ADVERTS_RSS_URL)
raw_xml

'<?xml version="1.0" encoding="utf-8"?>\n\n<rss version="2.0">\n\t<channel>\n\t<title>Adverts.ie - Recent Ads All 962,782 listings</title>\n\t<link>https://www.adverts.ie/for-sale/</link>\n\t<description>Adverts.ie - Recent Ads All 962,782 listings</description>\n\t<language>en</language>\n\n<item>\n\t<title>\'The Scamp\' by Constance Shaw</title>\n\t\t<price><![CDATA[50]]></price>\n\t\t\t<county><![CDATA[Dublin]]></county>\n\t\t\t<area><![CDATA[Swords]]></area>\n\t\t\t<image_url>https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpYS5hZHNpbWcuY29tIiwia2V5IjoiYWNhMmJlOGI4Y2QzMWI1MTAyMzQ4ODEwNDAxMDNmMmJjMGM0YjllNzdiZDJmMjQ3YjM2M2M5ZjZmMjE1MzAzMi5qcGciLCJvdXRwdXRGb3JtYXQiOiJqcGVnIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoyMjcsImhlaWdodCI6MTg3fX19?signature=b73cf3d6656f75d30f288be413a4b071d9e7e807b93d87dba50e89a8bfb44eda</image_url>\n\t\t<link>https://www.adverts.ie/pictures-paintings/the-scamp-by-constance-shaw/41164524</link>\t\n\t<guid>41164524</guid>\n\t<pubDate>Fri, 27 Nov 2026 11:42:01 +0000</p

    Item data-structure in XML Response is as below :-

        <item>\n\t
            <title>{{Title}}</title>\n\t\t
            <price><![CDATA[price]]></price>\n\t\t\t
            <county><![CDATA[county]]></county>\n\t\t\t
            <area><![CDATA[area]]></area>\n\t\t\t
            <image_url>{{Image}}</image_url>\n\t\t
            <link>{{Link}}</link>\t\n\t
            <guid>{{unique number}}</guid>\n\t
            <pubDate>{{Date}}</pubDate>\n\t
            <description>{{TEXT}}</description>\n
        </item>\n\n
    
    Note:-
    In XML, <![CDATA[SAMPLE]]> is a CDATA (Character Data) section that tells the XML parser to treat the word "SAMPLE" strictly as plain, literal text rather than markup.

### Part 2
Store the raw RSS feed into designated file and then parse the XML Response.

Folder structure for storing feed: YEAR -> MONTH -> DAY -> NOW.xml (One file per xml response is stored in day, month and year wise manner)

In [28]:
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
import xml.etree.ElementTree as ET

In [29]:
RAW_DIR = Path("data/raw")

In [30]:
def save_raw_xml(xml_bytes: bytes, base_dir: Path = RAW_DIR) -> Path:
    """Save the response as data/raw/<year>/<month>/<day>/<timestamp>.xml"""
    now = datetime.now(timezone.utc)
    folder = base_dir / f"{now:%Y}" / f"{now:%m}" / f"{now:%d}"
    folder.mkdir(parents=True, exist_ok=True)

    file_path = folder / f"{now:%Y-%m-%d_%H%M%S}.xml"
    file_path.write_bytes(xml_bytes)
    return file_path


In [31]:
# AI Assisted Functions below :-
def get_text(item, tag):
    """Return the text inside a tag, or None if it is missing or empty."""
    element = item.find(tag)
    if element is not None and element.text:
        return element.text.strip()
    return None


def parse_date(value):
    """Convert an RSS date string into a datetime."""
    if value is None:
        return None
    return parsedate_to_datetime(value)


def parse_rss(xml_bytes: bytes) -> list[dict]:
    """Turn the RSS XML into a list of listing dictionaries."""
    root = ET.fromstring(xml_bytes)
    listings = []

    for item in root.iter("item"):
        listings.append({
            "area": get_text(item, "area"),
            "guid": get_text(item, "guid"),
            "price": get_text(item, "price"),
            "title": get_text(item, "title"),
            "county": get_text(item, "county"),
            "category": get_text(item, "category"),
            "image_url": get_text(item, "image_url"),
            "description": get_text(item, "description"),
            "published_at": parse_date(get_text(item, "pubDate")),
        })
    return listings

In [32]:
saved_path = save_raw_xml(raw_xml.encode("utf-8")) # converts into bytes
print(saved_path)

data\raw\2026\11\27\2026-11-27_114209.xml


In [33]:
listings = parse_rss(raw_xml)
listings

[{'area': 'Swords',
  'guid': '41164524',
  'price': '50',
  'title': "'The Scamp' by Constance Shaw",
  'county': 'Dublin',
  'category': None,
  'image_url': 'https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpYS5hZHNpbWcuY29tIiwia2V5IjoiYWNhMmJlOGI4Y2QzMWI1MTAyMzQ4ODEwNDAxMDNmMmJjMGM0YjllNzdiZDJmMjQ3YjM2M2M5ZjZmMjE1MzAzMi5qcGciLCJvdXRwdXRGb3JtYXQiOiJqcGVnIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoyMjcsImhlaWdodCI6MTg3fX19?signature=b73cf3d6656f75d30f288be413a4b071d9e7e807b93d87dba50e89a8bfb44eda',
  'description': '<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a href="https://www.adverts.ie/pictures-paintings/the-scamp-by-constance-shaw/41164524">\n\t\t\t\t\t\t\t<img src="https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpYS5hZHNpbWcuY29tIiwia2V5IjoiYWNhMmJlOGI4Y2QzMWI1MTAyMzQ4ODEwNDAxMDNmMmJjMGM0YjllNzdiZDJmMjQ3YjM2M2M5ZjZmMjE1MzAzMi5qcGciLCJvdXRwdXRGb3JtYXQiOiJqcGVnIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoyMjcsImhlaWdodCI6MTg3fX19?signature=b73cf3d6656f75d30f288be413a4b071d9e7e807b93d87dba50e89a8bfb44

In [34]:
df = pd.DataFrame(listings)
df.head()

,area,guid,price,title,county,category,image_url,description,published_at
0,Swords,41164524,50,'The Scamp' by Constance Shaw,Dublin,None,https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpY...,<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a h...,2026-11-27 11:42:01+00:00
1,Dublin 1,29133695,15,Nfl Street Xbox Pal version games Original,Dublin,None,https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpY...,<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a h...,2026-11-27 11:42:00+00:00
2,Dublin 1,29133981,12,Halo 2 Xbox Pal version games Original,Dublin,None,https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpY...,<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a h...,2026-11-27 11:42:00+00:00
3,Dublin 1,29134402,9,Rainbow six 3 Tom Clancy s Xbox Original Pal v...,Dublin,None,https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpY...,<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a h...,2026-11-27 11:42:00+00:00
4,Dublin 1,30134457,15,Aeon Flux xbox Original,Dublin,None,https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpY...,<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a h...,2026-11-27 11:42:00+00:00


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   area          30 non-null     str                
 1   guid          30 non-null     str                
 2   price         30 non-null     str                
 3   title         30 non-null     str                
 4   county        30 non-null     str                
 5   category      0 non-null      object             
 6   image_url     30 non-null     str                
 7   description   30 non-null     str                
 8   published_at  30 non-null     datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), object(1), str(7)
memory usage: 2.2+ KB


In [36]:
df.describe()

,area,guid,price,title,county,category,image_url,description,published_at
count,30,30,30,30,30,0,30,30,30
unique,10,30,13,30,2,0,30,30,NaN
top,Dublin 1,41164524,12,'The Scamp' by Constance Shaw,Dublin,NaN,https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpY...,<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a h...,NaN
freq,17,1,7,1,29,NaN,1,1,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-11-27 11:41:46.500000+00:00
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-11-27 11:41:25+00:00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-11-27 11:41:43+00:00
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-11-27 11:41:44+00:00
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-11-27 11:41:59+00:00
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-11-27 11:42:01+00:00


# Connecting the Database

In [37]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [ ]:
load_dotenv()

# Run in powershell: ssh -i ~/.ssh/gcp_vm -L 5433:localhost:5432 -N abhishek@35.226.8.82
url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.environ["DB_HOST"],
    port=int(os.environ["DB_PORT"]),
    database=os.environ["DB_NAME"],
)
engine = create_engine(url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.scalar())  # First Column of the first Row

PostgreSQL 18.6 (Ubuntu 18.6-0ubuntu0.26.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 15.2.0-16ubuntu1) 15.2.0, 64-bit


TODO:
1. Need to clean "description" column in the item response and extract useful text
2. Design appropriate Database Table schema and Entities and save the response
3. guid is the unique identifier so will use it to identify records with same guid before saving any item
4. Download and save the image into appropriate directories and save them with unique identifier which will be assigned to each item
5. Setup cron for this process (every 2-3 minutes)